# Worksheet 5: Decode with the correct key by applying $F^\dagger D_k^\dagger$

**Task 5 of 5.** Bob receives $|\psi_{m,k}\rangle$, knows $k$, and wants $m$ back.

```
 Alice: |0…0⟩ ─[X]─[F]─[D_k]─► |ψ_{m,k}⟩ ─► Bob: ─[ ? ]─[ ? ]─[measure]─► m
```

What you'll do, in this order:
1. work out the inverse on paper (and the order of the gates)
2. choose gates for $D_k^\dagger$ and $F^\dagger$
3. build Bob's decoder
4. build the full Alice → Bob circuit
5. decode one message
6. prove decoder ∘ encoder = identity
7. run all $M\cdot K$ pairs
8. (optional) shots like a real device
9. break it on purpose: four classic bugs

### How to use this worksheet
* **📖 Read** cells explain a concept. **✍️ On paper** cells ask you to work something out by hand first.
* **✏️ TODO** cells are yours: replace `None` or the `raise NotImplementedError` with your own code.
* **✅ Check** cells test what you wrote. They print ✅ (correct), ❌ (what is wrong, not how to fix it) or ⏳ (not done yet). **Don't move on until you get ✅.**
* **💡 Hints** are folded. Open them only if you are stuck; Hint 1 is gentle, the last one names the exact function or gate.
* **🔀 Options** tables list the different ways to do a step. Choose one, and write your choice and reason in the decision log at the end, which you can reuse in the report.

## Setup

In [ ]:
# Setup (given — nothing to solve here)
# If a package is missing:  !pip install qiskit matplotlib
import numpy as np
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, Operator, DensityMatrix
from qiskit.circuit.library import QFTGate, DiagonalGate, UnitaryGate

from lock_checks import *     # the automatic checks + plot_circles (keep lock_checks.py in this folder)

np.set_printoptions(precision=3, suppress=True, linewidth=130)
print("Qiskit", qiskit.__version__, "ready")

## Paste your work from worksheets 1–4

In [ ]:
# ✏️ paste
N_DEMO, N = None, None
d_demo, d, M = None, None, None
K, SEED, MODE = None, None, None

def bits_of(m, n):
    raise NotImplementedError

def ket(m, d):
    raise NotImplementedError

def encode_message(m, n):
    raise NotImplementedError

def dft_matrix(n):
    raise NotImplementedError

def make_codebook(n, K, seed=2026, mode="uniform"):
    raise NotImplementedError

def D_matrix(theta_row):
    raise NotImplementedError

def D_gate(theta_row):
    raise NotImplementedError

def lock_circuit(m, k, n, thetas):
    raise NotImplementedError

In [ ]:
check_encode(encode_message); check_dft(dft_matrix); check_codebook(make_codebook); check_D_gate(D_gate)
try:
    thetas      = make_codebook(N, K, SEED, MODE)
    thetas_demo = make_codebook(N_DEMO, K, SEED, MODE)
    check_lock_circuit(lock_circuit, thetas)
except NotImplementedError:
    thetas = thetas_demo = None
    print('⏳ paste your worksheet functions above first')

---
## Step 5.1: On paper: the inverse, and the order

✍️ Alice applied $U_{\text{lock}}=D_kF$.
1. What is $(AB)^{-1}$ in terms of $A^{-1}$ and $B^{-1}$? (Think "socks and shoes".)
2. For a unitary $U$, what is $U^{-1}$?
3. So $(D_kF)^{-1} = \;?$
4. Verify: write $(D_kF)^{-1}D_kF|m\rangle$ and simplify step by step down to $|m\rangle$.
5. In a **circuit diagram** (left to right), which of Bob's two gates comes **first**?

📖 **Bob's measurement.** Undoing the lock and then measuring in the computational basis is the same as measuring in the basis $\{|\psi_{m',k}\rangle\}_{m'}$ of codewords for key $k$. That is the **key-dependent decoding measurement** of the paper (Sec. II, the POVM $\Lambda_m^{(k)}$, here simply projectors). Paper bullet: *"How Bob uses the key to select the correct decoding measurement."*

## Step 5.2: Gates for $D_k^\dagger$ and $F^\dagger$

🔀 **Options for $D_k^\dagger$**

| Option | Idea | Comment |
|---|---|---|
| A | a new diagonal gate with conjugated entries | think: what are the angles of $D_k^\dagger$? |
| B | `.inverse()` of your `D_gate` | let Qiskit do it |
| C | `UnitaryGate` of the conjugate-transposed matrix | works, expensive |

🔀 **Options for $F^\dagger$**

| Option | Idea | Comment |
|---|---|---|
| A | `.inverse()` of the QFT gate | recommended in the brief |
| B | `.inverse()` of your `qft_manual` circuit: Qiskit reverses the gate list and negates the angles (course: invQFT) | instructive |
| C | `UnitaryGate` of $F^\dagger$ | expensive |

Build both gates for key `k = 2`, project size.

In [ ]:
# ✏️ TODO 5.2
k = 2
Ddag_gate = None
Fdag_gate = None

In [ ]:
check_dagger_gates(Ddag_gate, Fdag_gate, thetas[k] if thetas is not None else None, N)

<details><summary>💡 Hint 1</summary>

$D_k^\dagger=\mathrm{diag}(e^{-i\theta_{k,x}})$, so which angles do you pass to your `D_gate`?

</details>

<details><summary>💡 Hint 2</summary>

`D_gate(-thetas[k])` or `D_gate(thetas[k]).inverse()`; `QFTGate(N).inverse()`.

</details>

## Step 5.3: Bob's decoder circuit

Write `decode_circuit(k, n, thetas)`: an $n$-qubit circuit with Bob's two gates **in the right order** (your answer to 5.1 question 5).

In [ ]:
# ✏️ TODO 5.3
def decode_circuit(k, n, thetas):
    raise NotImplementedError

In [ ]:
check_decoder(decode_circuit, thetas_demo)
check_decoder(decode_circuit, thetas)

## Step 5.4: The full Alice → channel → Bob circuit

Write `full_circuit(m, k_alice, k_bob, n, thetas)`:
* Alice locks $m$ with `k_alice` (your `lock_circuit`)
* a `barrier` labelled "channel" (optional, for the drawing)
* Bob decodes with `k_bob`
* **special case `k_bob=None`** means **no key**: Bob/Eve applies only the public $F^\dagger$, with no phase correction. The brief asks you to test this case in tasks 6–7.

Having `k_bob` separate from `k_alice` already prepares task 6 (wrong key).

In [ ]:
# ✏️ TODO 5.4
def full_circuit(m, k_alice, k_bob, n, thetas):
    raise NotImplementedError

In [ ]:
check_full(full_circuit, thetas)

<details><summary>💡 Hint 1</summary>

`qc = lock_circuit(...)`, then `qc.compose(other_circuit, inplace=True)` adds Bob's part.

</details>

In [ ]:
# ✏️ draw full_circuit(5, 2, 2, N_DEMO, thetas_demo)

## Step 5.5: Decode one message

Run `full_circuit(37, 2, 2, N, thetas)`, print the (non-zero) probabilities and turn the most likely bitstring into an integer, like in worksheet 1 Step 1.8.

Then draw the five stages at demo size with `plot_circles` ($|m\rangle$ → $F$ → $D_k$ → $D_k^\dagger$ → $F^\dagger$). ✍️ What symmetry do you see?

In [ ]:
# ✏️ TODO 5.5
m_decoded = None

In [ ]:
check_readout(m_decoded, 37)

## Step 5.6: The strongest check: encoder ∘ decoder = identity

📖 Testing states for a few $m$ is good. Testing the **whole operator** proves it for every input at once: $F^\dagger D_k^\dagger D_kF=I$ for every key.

🔀 **Options**

| Option | How |
|---|---|
| A | NumPy: multiply your four matrices and compare with `np.eye(d)` |
| B | Qiskit: a circuit **without** the X gates (QFT, $D_k$, then your decoder), `Operator(circuit).data` vs identity |

Do it for every key. (Doing both A and B is a good cross-check.)

In [ ]:
# ✏️ TODO 5.6: print, for each key, whether the operator is the identity

## Step 5.7: Run the whole test set

For every $(m,k)$ (all $M\cdot K$), simulate `full_circuit(m, k, k, N, thetas)` and store a dict with `"m"`, `"k"`, `"p_recover"` (the probability of outcome $m$) and `"decoded"` (the most likely outcome). Print the average and minimum of `p_recover`.

Then do the same with a **binary** codebook. Same result?

In [ ]:
# ✏️ TODO 5.7
results = None

In [ ]:
check_results(results, M, K)

<details><summary>💡 Hint 1</summary>

`Statevector.from_instruction(qc).probabilities()` gives an array indexed by the integer outcome, which is easier here than the dict.

</details>

## Step 5.8 (optional): Shots, like a real device

🔀 **Options**

| Option | How | Comment |
|---|---|---|
| A | `sv.sample_counts(shots)` on your `Statevector` (use `sv.seed(...)` for repeatability) | no extra install |
| B | a classical register + `measure` (worksheet 1, option B) and `qiskit_aer.AerSimulator` or a `Sampler` | what you'd do for hardware |

✍️ Predict: with the correct key and 1024 shots, how many different outcomes will you see?

In [ ]:
# ✏️ TODO 5.8 (optional)

## Step 5.9: Break it on purpose: four classic bugs

For $m=37$, $k=2$, build Bob's side **wrongly** in each of these ways and look at $P(m)$ and the most likely outcome. ✍️ **Predict each result first.**

| Bug | Bob applies | Your prediction | Result |
|---|---|---|---|
| 1. wrong order | $F^\dagger$ first, then $D_k^\dagger$ | | |
| 2. forgot the dagger on $D$ | $D_k$, then $F^\dagger$ | | |
| 3. used $F$ instead of $F^\dagger$ | $D_k^\dagger$, then $F$ | | |
| 4. read the bitstring backwards | correct gates, `int(bits[::-1], 2)` | | |

Then repeat bug 2 with a **binary** codebook. ✍️ Why does it "work" there? (Look back at worksheet 3, Step 3.6b.) Why is that dangerous when testing?

For bug 3, compare the outcome with $-m \bmod d$ and with worksheet 2, Step 2.4.

In [ ]:
# ✏️ TODO 5.9

## Step 5.10: Warm-up for tasks 6–7: entropy

📖 Shannon entropy $H(p)=-\sum_x p_x\log_2p_x$ measures how spread out a distribution is: 0 bits when the outcome is certain, $\log_2 d$ bits when it's uniform. You'll use it to compare correct-key, wrong-key and no-key decoding.

Write `entropy_bits(p)`. Careful: $0\cdot\log 0$ must count as 0.

In [ ]:
# ✏️ TODO 5.10
def entropy_bits(p):
    raise NotImplementedError

In [ ]:
check_entropy(entropy_bits)

🔬 **Sneak peek (optional):** for $m=37$, $k=2$, compute the output distribution and its entropy with (a) the correct key, (b) a wrong key, (c) `k_bob=None`. Plot the three as bar charts side by side. That is the start of tasks 6–7.

In [ ]:
# ✏️ optional sneak peek

---
## For your report (paper bullets you can now answer)
Write these in **your own words**, using what you measured:
* **How Bob uses the key:** …
* **Bob's decoding probability vs Eve's accessible information:** Bob's $\bar p_{succ}$ (paper Eq. 3) is about one specific key-dependent measurement. What is Eve's $I_{acc}$ (Eq. 22) maximized over? Why do your wrong-key / no-key tests **not** prove security? (The brief says so explicitly.)
* **Simplification vs the paper:** what is general in the paper (random phases, large $d$, POVMs for the first protocol, the bound in Theorem 4) and what is specific in your circuit ($d=64$, $K=4$, exact inverse unitary)?

## Decision log

| Step | Options | My choice | Why |
|---|---|---|---|
| 5.2 $D_k^\dagger$ | conjugated diagonal / `.inverse()` / UnitaryGate | | |
| 5.2 $F^\dagger$ | `QFTGate.inverse()` / manual inverse / UnitaryGate | | |
| 5.6 check | NumPy / Operator | | |
| 5.8 readout | exact / sample_counts / Aer | | |

✅ All green? Tasks 1–5 are done. Next come tasks 6–8: wrong keys, no key, distributions, entropies, and averages over all $(m,k)$ and different $K$, $n$.